
# NBA MVPs  Web Scrape & Analysis with BeautifulSoup





In [ ]:

# If needed, uncomment and run this to install dependencies.
# %pip install requests beautifulsoup4 lxml pandas matplotlib


In [1]:

import re
import math
import requests
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

WIKI_URL = "https://en.wikipedia.org/wiki/NBA_Most_Valuable_Player_Award"


In [ ]:

def fetch_html(url: str) -> str:
    """Fetch page HTML (simple GET)."""
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.text


def find_mvp_table(soup: BeautifulSoup):
    """
    Locate the main 'Winners' table by checking for headers that include
    'Season' and 'Player' and either 'Team' or 'Team(s)'.
    """
    for table in soup.select("table.wikitable"):
        headers = [th.get_text(strip=True) for th in table.select("thead th")] or                   [th.get_text(strip=True) for th in table.select("tr th")]
        headers_lower = [h.lower() for h in headers]
        if ("season" in " ".join(headers_lower) and
            "player" in " ".join(headers_lower) and
            ("team" in " ".join(headers_lower) or "team(s)" in " ".join(headers_lower))):
            return table
    return None


def normalize_position(raw: str) -> str:
    """
    Normalize raw position text to base buckets: 'guard', 'forward', 'center'.
    Handles abbreviations and mixed positions like 'G-F', 'F/C', 'Forward-Center' etc.
    """
    if not raw:
        return ""
    s = raw.strip().lower()
    # Replace separators with a uniform slash
    s = re.sub(r"[–—-]", "/", s)  # en/em dashes to slash
    s = s.replace("\\", "/")

    # Word-level normalization
    s = s.replace("point guard", "guard")
    s = s.replace("shooting guard", "guard")
    s = s.replace("small forward", "forward")
    s = s.replace("power forward", "forward")
    s = s.replace("pf", "forward").replace("sf", "forward")
    s = s.replace("pg", "guard").replace("sg", "guard")
    s = s.replace("c/f", "center/forward").replace("f/c", "forward/center")
    s = s.replace("g/f", "guard/forward").replace("f/g", "forward/guard")

    # Single-letter short forms
    s = re.sub(r"\b[gc]\b", lambda m: "guard" if m.group(0) == "g" else "center", s)
    s = re.sub(r"\bf\b", "forward", s)

    # Keep only our three buckets words
    parts = re.split(r"[ /,]+", s)
    parts = [p for p in parts if p in {"guard", "forward", "center"}]

    # Return canonical joined (for possible combo, we keep "guard/forward" etc.)
    parts = sorted(set(parts), key=lambda x: ["guard", "forward", "center"].index(x))
    return "/".join(parts)


def split_position_weights(pos: str):
    """
    Convert a normalized position string like 'guard/forward' into weighted parts.
    Returns list of (bucket, weight) summing to 1.0.
    """
    if not pos:
        return []
    parts = pos.split("/")
    if not parts:
        return []
    w = 1.0 / len(parts)
    return [(p, w) for p in parts]


In [ ]:

html = fetch_html(WIKI_URL)
soup = BeautifulSoup(html, "lxml")

table = find_mvp_table(soup)
if table is None:
    raise RuntimeError("Could not locate the MVP 'Winners' table on the page.")

# Identify header indices
headers = [th.get_text(strip=True) for th in table.select("tr th")]
headers_lower = [h.lower() for h in headers]

def header_index(names):
    for n in names:
        if n in headers_lower:
            return headers_lower.index(n)
    return None

season_idx = header_index(["season"])
player_idx = header_index(["player"])
team_idx   = header_index(["team(s)", "team", "club"])
pos_idx    = header_index(["position", "pos.", "pos"])

rows = []
for tr in table.select("tr"):
    tds = tr.find_all(["td", "th"])
    if not tds or len(tds) < 3:
        continue
    try:
        season = tds[season_idx].get_text(" ", strip=True) if season_idx is not None else ""
        player = tds[player_idx].get_text(" ", strip=True) if player_idx is not None else ""
        team   = tds[team_idx].get_text(" ", strip=True)   if team_idx   is not None else ""
        pos_raw = tds[pos_idx].get_text(" ", strip=True)   if pos_idx    is not None else ""
    except Exception:
        continue

    # Basic guards to skip header or malformed rows
    if not season or season.lower() == "season" or not player or player.lower() == "player":
        continue

    rows.append({
        "season": season,
        "player": player,
        "position_raw": pos_raw,
        "position_norm": normalize_position(pos_raw),
        "team": team
    })

mvp_df = pd.DataFrame(rows).reset_index(drop=True)
mvp_df.head()


In [ ]:

def clean_team_name(name: str) -> str:
    if not name:
        return name
    s = name.strip()
    # Remove footnote markers like [a], [1], etc.
    s = re.sub(r"\[\w+\]|\[\d+\]", "", s).strip()
    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s)
    return s

mvp_df["team"] = mvp_df["team"].map(clean_team_name)
mvp_df.head()


In [ ]:

# Weighted counting for combo positions (e.g., 'guard/forward' counts 0.5 to each)
pos_buckets = {"guard": 0.0, "forward": 0.0, "center": 0.0}

for pos in mvp_df["position_norm"]:
    parts = split_position_weights(pos)
    if not parts:
        # If missing, skip
        continue
    for bucket, w in parts:
        if bucket in pos_buckets:
            pos_buckets[bucket] += w

pos_series = pd.Series(pos_buckets).sort_values(ascending=False)
print("MVP awards by position (weighted for combos):")
print(pos_series)

# Bar chart (no custom colors; single plot)
plt.figure()
pos_series.plot(kind="bar")
plt.title("NBA MVP Awards by Position (weighted)")
plt.xlabel("Position")
plt.ylabel("Awards (weighted count)")
plt.tight_layout()
plt.show()


In [ ]:

team_counts = (mvp_df.assign(team=mvp_df["team"].fillna(""))
               .loc[lambda d: d["team"] != ""]
               .groupby("team")["season"].count()
               .sort_values(ascending=False))

print("Teams with the most MVP awards (top 12):")
print(team_counts.head(12))

plt.figure()
team_counts.head(12).plot(kind="bar")
plt.title("Top Teams by NBA MVP Awards")
plt.xlabel("Team")
plt.ylabel("Number of MVP Awards")
plt.tight_layout()
plt.show()


In [ ]:

# Optional: save the parsed table to CSV for reuse
mvp_df.to_csv("nba_mvp_winners.csv", index=False)
print("Saved parsed data to nba_mvp_winners.csv")


In [ ]:

from IPython.display import display
display(mvp_df.head(20))
print(f"Total seasons parsed: {len(mvp_df)}")
